# Punto 7 · Analítica descriptiva con PySpark en EMR (JupyterHub)

Este notebook ejecuta, celda por celda, la misma lógica de `07-pyspark/analitica_pyspark.py`
(el script que corre en Glue y como step de EMR). Sirve para mostrar en la sustentación cómo se
construye cada respuesta y ver los resultados intermedios.

**Cómo abrirlo:** JupyterHub del EMR (`https://<master>:9443`, usuario `jovyan`, clave `jupyter`) →
*Upload* este archivo → kernel **PySpark**. La variable `spark` ya existe en ese kernel.

**Antes:** el workflow de Glue ya debe haber llenado `trusted/` (scripts/ejecutar_pipeline.sh).

## 1. Configuración: escriba el nombre de su bucket (output `bucket_datalake` de Terraform)

In [ ]:
BUCKET = "retail-p2-datalake-CAMBIAR-POR-SU-CUENTA"
TRUSTED = f"s3://{BUCKET}/trusted"
spark.conf.set("spark.sql.session.timeZone", "UTC")
print(spark.version, TRUSTED)

## 2. Las 5 tablas de la zona trusted (Parquet)
Cada una viene de una fuente distinta: ventas (EC2), clientes y productos (RDS), países (RDS + Banco Mundial), tasas de cambio (Frankfurter).

In [ ]:
for tabla in ["ventas", "clientes", "productos", "paises", "tasas_cambio"]:
    df = spark.read.parquet(f"{TRUSTED}/{tabla}")
    print(f"{tabla}: {df.count()} filas")
    df.printSchema()

## 3. Vista enriquecida: une las 3 fuentes
ventas ⨝ clientes ⨝ países ⨝ productos ⨝ tasas de cambio, y calcula `importe_usd`.

In [ ]:
from pyspark.sql import functions as F

DIAS = {1: "Lunes", 2: "Martes", 3: "Miércoles", 4: "Jueves", 5: "Viernes", 6: "Sábado", 7: "Domingo"}

def ventas_enriquecidas(spark, trusted):
    """Une las 3 fuentes en una sola vista de trabajo:
         ventas (EC2) + clientes/países/productos (RDS) + tasas y población (URL)."""
    ventas = spark.read.parquet(f"{trusted}/ventas")
    clientes = spark.read.parquet(f"{trusted}/clientes")
    paises = spark.read.parquet(f"{trusted}/paises")
    productos = spark.read.parquet(f"{trusted}/productos")
    tasas = spark.read.parquet(f"{trusted}/tasas_cambio")
    return (ventas
            .join(clientes, "id_cliente")
            .join(paises, "pais")
            .join(productos, "codigo_producto")
            .join(tasas, "fecha")
            .withColumn("importe_usd", F.col("importe_gbp") * F.col("gbp_usd")))

def solo_ventas(col):
    """Suma/cuenta solo las líneas que NO son devolución."""
    return F.when(~F.col("es_devolucion"), col)

v = ventas_enriquecidas(spark, TRUSTED).cache()
print("líneas enriquecidas:", v.count())
v.select("factura", "pais", "region_comercial", "descripcion", "cantidad", "importe_gbp", "gbp_usd", "importe_usd").show(5, truncate=False)

## P1. ¿Cuáles son los 10 países con mayor ingreso neto en dólares?

In [ ]:
def p1_top_paises_ingreso_usd(v):
    """P1. ¿Cuáles son los 10 países con mayor ingreso neto en dólares?"""
    return (v.groupBy("pais")
             .agg(F.round(F.sum("importe_usd"), 2).alias("ingreso_neto_usd"),
                  F.round(F.sum("importe_gbp"), 2).alias("ingreso_neto_gbp"),
                  F.countDistinct("id_cliente").alias("clientes"))
             .orderBy(F.desc("ingreso_neto_usd"))
             .limit(10))

p1_top_paises_ingreso_usd(v).show(100, truncate=False)

## P2. ¿Cómo evolucionó el ingreso mes a mes y cuál fue el mes pico?

In [ ]:
def p2_ingreso_mensual(v):
    """P2. ¿Cómo evolucionó el ingreso mes a mes y cuál fue el mes pico?"""
    return (v.groupBy(F.year("fecha").alias("anio"), F.month("fecha").alias("mes"))
             .agg(F.round(F.sum("importe_gbp"), 2).alias("ingreso_neto_gbp"),
                  F.round(F.sum("importe_usd"), 2).alias("ingreso_neto_usd"),
                  F.countDistinct(solo_ventas(F.col("factura"))).alias("facturas"),
                  F.round(F.sum(solo_ventas(F.col("importe_gbp")))
                          / F.countDistinct(solo_ventas(F.col("factura"))), 2).alias("ticket_promedio_gbp"))
             # "2011-03": etiqueta lista para el eje X de las gráficas (punto 8)
             .withColumn("periodo", F.format_string("%d-%02d", F.col("anio"), F.col("mes")))
             .orderBy("anio", "mes"))

p2_ingreso_mensual(v).show(100, truncate=False)

## P3. ¿Cuáles son los 10 productos que más ingreso neto generan?

In [ ]:
def p3_top_productos(v):
    """P3. ¿Cuáles son los 10 productos que más ingreso neto generan?"""
    return (v.groupBy("codigo_producto", "descripcion")
             .agg(F.sum("cantidad").alias("unidades_netas"),
                  F.round(F.sum("importe_gbp"), 2).alias("ingreso_neto_gbp"),
                  F.round(F.sum("importe_usd"), 2).alias("ingreso_neto_usd"))
             .orderBy(F.desc("ingreso_neto_gbp"))
             .limit(10))

p3_top_productos(v).show(100, truncate=False)

## P4. ¿En qué países se gasta más en la tienda por cada 1.000 habitantes?

In [ ]:
def p4_gasto_por_habitante(v):
    """P4. ¿En qué países se gasta más en la tienda por cada 1.000 habitantes?"""
    return (v.filter(F.col("poblacion").isNotNull())
             .groupBy("pais", "poblacion", "pib_per_capita_usd")
             .agg(F.round(F.sum("importe_usd"), 2).alias("ingreso_neto_usd"),
                  F.round(F.sum("importe_usd") / F.first("poblacion") * 1000, 4).alias("usd_por_mil_habitantes"))
             .select("pais", "poblacion", "pib_per_capita_usd", "ingreso_neto_usd", "usd_por_mil_habitantes")
             .orderBy(F.desc("usd_por_mil_habitantes"))
             .limit(10))

p4_gasto_por_habitante(v).show(100, truncate=False)

## P5. ¿Qué región comercial devuelve más (en % del valor vendido)?

In [ ]:
def p5_devoluciones_por_region(v):
    """P5. ¿Qué región comercial devuelve más (en % del valor vendido)?"""
    brutas = F.sum(F.when(~F.col("es_devolucion"), F.col("importe_gbp")).otherwise(0))
    devueltas = -1 * F.sum(F.when(F.col("es_devolucion"), F.col("importe_gbp")).otherwise(0))
    return (v.groupBy("region_comercial")
             .agg(F.round(brutas, 2).alias("ventas_brutas_gbp"),
                  F.round(devueltas, 2).alias("devoluciones_gbp"),
                  F.round(100 * devueltas / brutas, 2).alias("tasa_devolucion_pct"),
                  F.countDistinct(F.when(F.col("es_devolucion"), F.col("factura"))).alias("facturas_devueltas"))
             .orderBy(F.desc("tasa_devolucion_pct")))

p5_devoluciones_por_region(v).show(100, truncate=False)

## P6. ¿Qué días de la semana y a qué horas se concentran las compras?

In [ ]:
def p6_ventas_dia_hora(v):
    """P6. ¿Qué días de la semana y a qué horas se concentran las compras?"""
    nombre_dia = F.create_map(*[x for k, n in DIAS.items() for x in (F.lit(k), F.lit(n))])
    return (v.filter(~F.col("es_devolucion"))
             .groupBy("dia_semana", "hora")
             .agg(F.countDistinct("factura").alias("facturas"),
                  F.round(F.sum("importe_gbp"), 2).alias("ingreso_bruto_gbp"))
             .withColumn("dia", nombre_dia[F.col("dia_semana")])
             .select("dia_semana", "dia", "hora", "facturas", "ingreso_bruto_gbp")
             .orderBy("dia_semana", "hora"))

p6_ventas_dia_hora(v).show(100, truncate=False)

## 4. Guardar la zona refined desde EMR
Mismo resultado que el job de Glue `trusted-a-refined`: sobrescribe `refined/` y el crawler refined las cataloga.

In [ ]:
for pregunta in [p1_top_paises_ingreso_usd, p2_ingreso_mensual, p3_top_productos,
                 p4_gasto_por_habitante, p5_devoluciones_por_region, p6_ventas_dia_hora]:
    destino = f"s3://{BUCKET}/refined/{pregunta.__name__}"
    pregunta(v).coalesce(1).write.mode("overwrite").parquet(destino)
    print("ok ->", destino)